In [ ]:
from pathlib import Path
import pandas as pd
if not Path('incident_event_log.csv').exists():
    from google.colab import files
    files.upload()
df = pd.read_csv('incident_event_log.csv', low_memory=False)
print('Dataset shape:',df.shape)
print('Unique incidents:',df['number'].nunique())
df.head()


In [ ]:
print('Dataset shape:', df.shape)
print('Total event records:', len(df))
print('Unique incidents:', df['number'].nunique())
print('Updates for INC0000045:', df['number'].value_counts().get('INC0000045', 0))


In [7]:
columns_to_view = [
    "number",
    "incident_state",
    "impact",
    "urgency",
    "priority",
    "reassignment_count",
    "opened_at",
    "sys_updated_at",
    "resolved_at"
]

df[columns_to_view].head(10)

,number,incident_state,impact,urgency,priority,reassignment_count,opened_at,sys_updated_at,resolved_at
0,INC0000045,New,2 - Medium,2 - Medium,3 - Moderate,0,29/2/2016 01:16,29/2/2016 01:23,29/2/2016 11:29
1,INC0000045,Resolved,2 - Medium,2 - Medium,3 - Moderate,0,29/2/2016 01:16,29/2/2016 08:53,29/2/2016 11:29
2,INC0000045,Resolved,2 - Medium,2 - Medium,3 - Moderate,0,29/2/2016 01:16,29/2/2016 11:29,29/2/2016 11:29
3,INC0000045,Closed,2 - Medium,2 - Medium,3 - Moderate,0,29/2/2016 01:16,5/3/2016 12:00,29/2/2016 11:29
4,INC0000047,New,2 - Medium,2 - Medium,3 - Moderate,0,29/2/2016 04:40,29/2/2016 04:57,1/3/2016 09:52
5,INC0000047,Active,2 - Medium,2 - Medium,3 - Moderate,1,29/2/2016 04:40,29/2/2016 05:30,1/3/2016 09:52
6,INC0000047,Active,2 - Medium,2 - Medium,3 - Moderate,1,29/2/2016 04:40,29/2/2016 05:33,1/3/2016 09:52
7,INC0000047,Active,2 - Medium,2 - Medium,3 - Moderate,1,29/2/2016 04:40,29/2/2016 11:31,1/3/2016 09:52
8,INC0000047,Active,2 - Medium,2 - Medium,3 - Moderate,1,29/2/2016 04:40,29/2/2016 11:32,1/3/2016 09:52
9,INC0000047,Active,2 - Medium,2 - Medium,3 - Moderate,1,29/2/2016 04:40,1/3/2016 09:14,1/3/2016 09:52


Primary columns selected:
*   number: Unique identifier assigned to an incident

*   incident_state:	Current state in that event record, such as New, Resolved or Closed

*   priority:	Operational priority assigned to the ticket

*   impact:	Assessment of how significantly the issue affects operations

*   urgency:	How quickly the issue needs attention

*   reassignment_count:	Number of times the ticket has been reassigned

*   made_sla:	Whether the incident met its configured service-level agreement target

*   opened_at:	When the incident was originally opened

*   sys_updated_at:	When that incident record was updated

*   resolved_at:	When the incident was recorded as resolved




In [8]:
incident = df[df["number"] == "INC0000045"]

incident[
    ["number", "incident_state", "opened_at", "sys_updated_at"]
]

,number,incident_state,opened_at,sys_updated_at
0,INC0000045,New,29/2/2016 01:16,29/2/2016 01:23
1,INC0000045,Resolved,29/2/2016 01:16,29/2/2016 08:53
2,INC0000045,Resolved,29/2/2016 01:16,29/2/2016 11:29
3,INC0000045,Closed,29/2/2016 01:16,5/3/2016 12:00


In [9]:
# Count missing values in every column
missing_values = df.isna().sum()

# Show the 10 columns with the most missing values
missing_values.sort_values(ascending=False).head(10)

,0
number,0
incident_state,0
active,0
reassignment_count,0
reopen_count,0
sys_mod_count,0
made_sla,0
caller_id,0
opened_by,0
opened_at,0


In [10]:
# Count cells containing the literal question-mark placeholder
question_marks = df.eq("?").sum()

# Display the 10 columns with the most question marks
question_marks.sort_values(ascending=False).head(10)

,0
caused_by,141689
vendor,141468
cmdb_ci,141267
rfc,140721
problem_id,139417
sys_created_by,53076
sys_created_at,53076
u_symptom,32964
assigned_to,27496
assignment_group,14213


### Task 1: Data Structure and Quality Analysis
We will inspect the column types, convert timestamps properly to calculate accurate missing percentages, look for exact duplicates, and analyze incident update distributions.

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make a copy to avoid modifying original df
eda_df = df.copy()

# 1. Identify all forms of missing values (?, empty, None, NaN)
# Replace '?' with actual NaN temporarily for calculations
eda_df_nan = eda_df.replace('?', np.nan)

missing_summary = pd.DataFrame({
    'Data Type': eda_df.dtypes,
    'Null/NaN Count': eda_df.isna().sum(),
    '"?" Count': eda_df.eq('?').sum(),
    'Blank String Count': eda_df.astype(str).apply(lambda x: x.str.strip().eq('')).sum()
})

missing_summary['Total Missing'] = missing_summary['Null/NaN Count'] + missing_summary['"?" Count'] + missing_summary['Blank String Count']
missing_summary['Missing %'] = (missing_summary['Total Missing'] / len(eda_df)) * 100

print("=== DATA STRUCTURE & MISSING VALUES REPORT ===")
display(missing_summary.sort_values(by='Total Missing', ascending=False))

# 2. Duplicate snapshot analysis
exact_duplicates = eda_df.duplicated().sum()
potential_duplicate_events = eda_df.duplicated(subset=['number', 'sys_updated_at']).sum()

print(f"\nExact duplicate rows (entire row matches): {exact_duplicates}")
print(f"Duplicate updates for the same incident at the same updated time: {potential_duplicate_events}")

=== DATA STRUCTURE & MISSING VALUES REPORT ===


,Data Type,Null/NaN Count,"""?"" Count",Blank String Count,Total Missing,Missing %
caused_by,object,0,141689,0,141689,99.983770
vendor,object,0,141468,0,141468,99.827820
cmdb_ci,object,0,141267,0,141267,99.685983
rfc,object,0,140721,0,140721,99.300694
problem_id,object,0,139417,0,139417,98.380518
sys_created_by,object,0,53076,0,53076,37.453427
sys_created_at,object,0,53076,0,53076,37.453427
u_symptom,object,0,32964,0,32964,23.261262
assigned_to,object,0,27496,0,27496,19.402732
assignment_group,object,0,14213,0,14213,10.029496



Exact duplicate rows (entire row matches): 0
Duplicate updates for the same incident at the same updated time: 14118


### Task 2: Exploratory Analysis & Visualizations
Let's investigate distributions of Priority, Impact, Urgency, SLA outcomes, and calculate Resolution Times (duration from `opened_at` to `resolved_at` for the final resolved state of each incident).

In [ ]:
# Helper: Parse datetime fields safely
time_cols = ['opened_at', 'sys_updated_at', 'resolved_at', 'closed_at']
for col in time_cols:
    eda_df_nan[col] = pd.to_datetime(eda_df_nan[col], format='%d/%m/%Y %H:%M', errors='coerce')

# Select the final snapshot or resolved snapshot of each ticket to evaluate overall metrics
resolved_tickets = eda_df_nan.assign(_revision=pd.to_numeric(eda_df_nan['sys_mod_count'], errors='coerce')).sort_values(['number', '_revision', 'sys_updated_at'], kind='stable', na_position='first').drop_duplicates('number', keep='last').copy()

# Calculate ticket resolution time in hours for resolved incidents
resolved_tickets['resolution_time_hours'] = (resolved_tickets['resolved_at'] - resolved_tickets['opened_at']).dt.total_seconds() / 3600.0

resolved_tickets = resolved_tickets[resolved_tickets['incident_state'].isin(['Resolved','Closed'])].copy()
resolved_tickets['resolution_time_hours'] = resolved_tickets['resolution_time_hours'].where(resolved_tickets['resolution_time_hours'].ge(0))
print("=== OPERATIONAL METRICS SUMMARY ===")
print(f"Total tickets analyzed: {resolved_tickets['number'].nunique()}")
print(f"Average resolution time (hours): {resolved_tickets['resolution_time_hours'].mean():.2f}")
print(f"Median resolution time (hours): {resolved_tickets['resolution_time_hours'].median():.2f}")

# 3. Create helpful plots
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Priority Distribution
sns.countplot(data=resolved_tickets, x='priority', ax=axes[0,0], order=resolved_tickets['priority'].value_counts().index, palette='Blues_r')
axes[0,0].set_title('Incident Priority Distribution')
axes[0,0].tick_params(axis='x', rotation=15)

# Plot 2: Reassignment count by Priority
sns.boxplot(data=resolved_tickets, x='priority', y='reassignment_count', ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Reassignments across Ticket Priorities')
axes[0,1].set_ylim(-1, 10)

# Plot 3: SLA compliance rate
sns.countplot(data=resolved_tickets, x='made_sla', ax=axes[1,0], palette='pastel')
axes[1,0].set_title('SLA Compliance (made_sla)')

# Plot 4: Resolution Times Distribution (capped at 120 hours for visualization quality)
sns.histplot(data=resolved_tickets[resolved_tickets['resolution_time_hours'] <= 120], x='resolution_time_hours', kde=True, ax=axes[1,1], color='purple')
axes[1,1].set_title('Resolution Time Distribution (Capped at 120h)')
axes[1,1].set_xlabel('Hours')

plt.tight_layout()
plt.show()

### Tasks 3, 4, and 5: Strategic Reporting Readiness & DORA Suitability Analysis
We will output code demonstrating how a robust incident-level transformation should aggregate key values before constructing our final summary.

In [13]:
# Quick aggregation check to answer Task 3
# Find incidents where sys_updated_at sequence ordering might be misaligned with incident status transitions
updates_per_incident = eda_df_nan.groupby('number').size()
print("=== EVENT-LEVEL DISTRIBUTION ===")
print(updates_per_incident.describe())


=== EVENT-LEVEL DISTRIBUTION ===
count    24918.000000
mean         5.687134
std          3.677845
min          2.000000
25%          3.000000
50%          5.000000
75%          7.000000
max         58.000000
dtype: float64


### Stage 1 Validation — Dataset Structure & Quality Audit
Let's run independent verification checks on the original `df` to double-check the dimensions, distinct incidents, exact duplicate rows, and multiple forms of missing values (including `?`, blanks, and true nulls).

In [14]:
# 1. Verify dimensions using shape and alternative count metrics
total_rows, total_cols = df.shape
unique_incidents_alt = len(df['number'].drop_duplicates())

print("--- Dimension Verification ---")
print(f"Verified Rows: {total_rows}")
print(f"Verified Columns: {total_cols}")
print(f"Verified Unique Incidents: {unique_incidents_alt}\n")

# 2. Check for exact duplicate rows vs. repeated incident IDs
# Entire row matching vs. Number of rows sharing the same incident ID
exact_dupes_verified = df.duplicated().sum()
legitimate_repeats = df.duplicated(subset=['number']).sum()

print("--- Duplication Verification ---")
print(f"Exact Duplicate Rows: {exact_dupes_verified}")
print(f"Legitimate Repeated Incident ID Entries (Events): {legitimate_repeats}\n")

# 3. Independent Missing Value Audit (Nulls, '?', Empty Strings)
# We look for true NaN values, literal '?' strings, and any whitespace-only cells
true_nulls = df.isna().sum()
question_marks = df.eq('?').sum()
whitespace_only = df.astype(str).apply(lambda x: x.str.strip().eq('')).sum()

total_missing_verified = true_nulls + question_marks + whitespace_only
percent_missing_verified = (total_missing_verified / total_rows) * 100

validation_df = pd.DataFrame({
    'True Nulls': true_nulls,
    '"?" Placeholders': question_marks,
    'Whitespace Only': whitespace_only,
    'Total Missing': total_missing_verified,
    'Missing %': percent_missing_verified
})

print("--- Top 10 Missing Value Columns Verified ---")
display(validation_df.sort_values(by='Total Missing', ascending=False).head(10))

--- Dimension Verification ---
Verified Rows: 141712
Verified Columns: 36
Verified Unique Incidents: 24918

--- Duplication Verification ---
Exact Duplicate Rows: 0
Legitimate Repeated Incident ID Entries (Events): 116794

--- Top 10 Missing Value Columns Verified ---


,True Nulls,"""?"" Placeholders",Whitespace Only,Total Missing,Missing %
caused_by,0,141689,0,141689,99.983770
vendor,0,141468,0,141468,99.827820
cmdb_ci,0,141267,0,141267,99.685983
rfc,0,140721,0,140721,99.300694
problem_id,0,139417,0,139417,98.380518
sys_created_by,0,53076,0,53076,37.453427
sys_created_at,0,53076,0,53076,37.453427
u_symptom,0,32964,0,32964,23.261262
assigned_to,0,27496,0,27496,19.402732
assignment_group,0,14213,0,14213,10.029496


### Task 3, 4, & 5: Validation Report & How to Validate Independently

#### 1. Task 1: Previous Claims Identification
- **Total Rows and Columns**: Reported as `141,712 × 36` in the exploratory cells (cell `A0Mdw8xAxqgU` and `iAZhhk_7zozG`). This was generated using `df.shape`.
- **Unique Incidents**: Reported as `24,918` unique incidents (cell `A0Mdw8xAxqgU` and `iAZhhk_7zozG`). This was calculated using `df["number"].nunique()`.
- **Missing-Value Counts**: Cells `k7wCm2qz2TQW` and `xBAFqY702Xog` identified that pandas standard `isna()` found 0 null values, while the literal placeholder string `"?"` occurred heavily in `caused_by` (141,689), `vendor` (141,468), `cmdb_ci` (141,267), `rfc` (140,721), etc.
- **Exact Duplicate Rows**: Cell `b1fe3135` reported `0` exact duplicate rows and `14,118` duplicated snapshots at the same timestamp (`sys_updated_at`) for the same incident.

#### 2. Task 3 & 4: Audit Validation Report

| Finding | Original AI Result | Independent Result | Status | Evidence |
| :--- | :--- | :--- | :--- | :--- |
| **Total Rows** | 141,712 | 141,712 | **VERIFIED** | `df.shape[0]` returns `141712`. |
| **Total Columns** | 36 | 36 | **VERIFIED** | `df.shape[1]` returns `36`. |
| **Unique Incidents** | 24,918 | 24,918 | **VERIFIED** | `len(df['number'].drop_duplicates())` matches. |
| **Exact Duplicate Rows** | 0 | 0 | **VERIFIED** | `df.duplicated().sum()` returns `0`. |
| **"?" in `caused_by`** | 141,689 | 141,689 | **VERIFIED** | `df['caused_by'].eq('?').sum()` returns `141689`. |
| **"?" in `vendor`** | 141,468 | 141,468 | **VERIFIED** | `df['vendor'].eq('?').sum()` returns `141468`. |
| **Whitespace / Empty Strings**| 0 across cols | 0 across cols | **VERIFIED** | Custom `.str.strip().eq('')` validation returns `0`. |

*No discrepancies were found between the initial automated check and the independent validation. Both correctly isolated literal `?` symbols from true Pandas Nulls.*

---

#### 3. Task 5: Teach Me to Validate Independently (Two Pro-Tips)

##### Lesson 1: The 'Hidden Missing Value' Trap
**Concept**: Modern libraries look for native Python `None` or NumPy `NaN` values when calling functions like `df.isna()` or `df.isnull()`. However, legacy systems and databases often export missing values as placeholder characters or strings (e.g., `'?'`, `'N/A'`, `'-'`, or empty spaces `''`).
- **How to validate**: Never trust `.isna()` alone! Always check unique values of object columns using `df['column_name'].unique()` or search for string lengths of 0 to catch hidden blanks.

##### Lesson 2: Verification by Triangulation
**Concept**: Never use the exact same formula to verify a result. If you calculated unique values using `df['number'].nunique()`, verify it by dropping duplicates and taking the length of the index: `len(df['number'].drop_duplicates())`.
- **Why we do this**: `nunique()` ignores true null values by default. Running an alternative method like `len(df['number'].unique())` might reveal if there are null values present in your identifier column that you did not account for. If the two methods mismatch, you have successfully exposed a silent bug!

### Prepare one whole latest snapshot per incident
Numeric ServiceNow revision count determines order, then update time and stable source order. Unknowns stay unknown. Ticket resolution time is not DORA incident duration or downtime. The five fictional scenarios remain separate from the public operational dataset.


In [ ]:
from pathlib import Path
Path('prepare_data.py').parent.mkdir(parents=True, exist_ok=True)
Path('prepare_data.py').write_text('"""Deterministic event-to-incident preparation; no regulatory metrics inferred."""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport pandas as pd\n\nTIME_COLS = [\'opened_at\',\'sys_updated_at\',\'resolved_at\',\'closed_at\']\nREQUIRED = [\'number\',\'incident_state\',\'sys_mod_count\',\'made_sla\',\'active\',\'priority\',\'impact\',\'urgency\',\'reassignment_count\'] + TIME_COLS\n\ndef boolean_series(s):\n    return s.astype(\'string\').str.strip().str.lower().map({\'true\':True,\'false\':False}).astype(\'boolean\')\n\ndef prepare(raw):\n    missing = sorted(set(REQUIRED)-set(raw.columns))\n    if missing:\n        raise ValueError(f\'Missing required columns: {missing}\')\n    p = raw.copy().replace(r\'^\\s*(?:\\?|)\\s*$\',pd.NA,regex=True)\n    if p[\'number\'].isna().any():\n        raise ValueError(\'Missing incident identifier\')\n    p[\'number\'] = p[\'number\'].astype(\'string\').str.strip()\n    p[\'_source_row\'] = range(len(p))\n    audit = {\'event_rows\':len(raw),\'source_columns\':len(raw.columns),\'exact_duplicate_rows\':int(raw.duplicated().sum()),\n             \'missing_by_column\':{k:int(v) for k,v in p[raw.columns].isna().sum().items()},\'invalid_datetimes\':{}}\n    for col in TIME_COLS:\n        original = p[col]\n        p[col] = pd.to_datetime(original,format=\'%d/%m/%Y %H:%M\',errors=\'coerce\')\n        audit[\'invalid_datetimes\'][col] = int((original.notna() & p[col].isna()).sum())\n    for col in [\'sys_mod_count\',\'reassignment_count\']:\n        p[col] = pd.to_numeric(p[col],errors=\'coerce\')\n    if p[\'sys_mod_count\'].isna().any() or (p.sys_mod_count < 0).any() or (p.sys_mod_count % 1 != 0).any():\n        raise ValueError(\'Missing/invalid revision count: cannot reliably select latest snapshot\')\n    for col in [\'made_sla\',\'active\']:\n        p[col] = boolean_series(p[col])\n    # ServiceNow sys_mod_count orders revisions; clock timestamps can be out of order.\n    # Preserve a whole snapshot (not groupby.last(), which splices non-null values).\n    ordered = p.sort_values([\'number\',\'sys_mod_count\',\'sys_updated_at\',\'_source_row\'],na_position=\'first\',kind=\'stable\')\n    out = ordered.drop_duplicates(\'number\',keep=\'last\').copy()\n    out[\'snapshot_tie\'] = out[\'number\'].isin(p.loc[p.duplicated([\'number\',\'sys_mod_count\',\'sys_updated_at\'],keep=False),\'number\'])\n    out[\'event_count\'] = out[\'number\'].map(p.groupby(\'number\').size())\n    duration = (out.resolved_at-out.opened_at).dt.total_seconds()/3600\n    out[\'duration_invalid\'] = duration.lt(0)\n    out[\'resolution_duration_hours\'] = duration.where(duration.ge(0) & out.incident_state.isin([\'Resolved\',\'Closed\']))\n    audit.update(incident_rows=len(out),unknown_sla=int(out.made_sla.isna().sum()),\n                 negative_durations=int(out.duration_invalid.sum()),missing_resolution_duration=int(out.resolution_duration_hours.isna().sum()),\n                 incidents_with_snapshot_ties=int(out.snapshot_tie.sum()),\n                 latest_states={str(k):int(v) for k,v in out.incident_state.value_counts(dropna=False).items()},\n                 mean_resolution_hours=float(out.resolution_duration_hours.mean()) if out.resolution_duration_hours.notna().any() else None,\n                 median_resolution_hours=float(out.resolution_duration_hours.median()) if out.resolution_duration_hours.notna().any() else None)\n    return out.drop(columns=\'_source_row\').reset_index(drop=True),audit\n\ndef build(path,output_dir):\n    path=Path(path); dest=Path(output_dir);dest.mkdir(parents=True,exist_ok=True)\n    raw=pd.read_csv(path,low_memory=False,keep_default_na=False)\n    out,audit=prepare(raw)\n    audit[\'raw_sha256\']=hashlib.sha256(path.read_bytes()).hexdigest()\n    out.to_csv(dest/\'incidents.csv\',index=False)\n    (dest/\'data_quality.json\').write_text(json.dumps(audit,indent=2)+\'\\n\')\n    print(json.dumps(audit,indent=2))\n    return out,audit\n\nif __name__==\'__main__\':\n    a=argparse.ArgumentParser();a.add_argument(\'source\');a.add_argument(\'--output-dir\',default=\'data\')\n    args=a.parse_args();build(args.source,args.output_dir)\n', encoding="utf-8")
Path('rules.py').parent.mkdir(parents=True, exist_ok=True)
Path('rules.py').write_text('"""Single-incident screening, EU 2024/1772 Arts 6,8,9; not a compliance decision."""\nfrom datetime import timedelta\nfrom decimal import Decimal\nimport math\nimport pandas as pd\n\nBOOL_FIELDS = {\n    \'critical_function\': \'ICT supporting a critical or important function affected (Art. 6(a))\',\n    \'regulated_service\': \'Authorised, registered or supervised financial service affected (Art. 6(b))\',\n    \'malicious_access\': \'Successful, malicious AND unauthorised system access (Art. 6(c))\',\n    \'access_may_cause_loss\': \'That access may result in data losses (Art. 9(5)(b))\',\n    \'relevant_clients\': \'Relevant clients or financial counterparts affected (Arts 1(3), 9(1)(f))\',\n    \'media\': \'Incident reflected in media (Art. 2(a))\',\n    \'complaints\': \'Repetitive complaints from different clients/counterparts on client-facing services or critical relationships (Art. 2(b))\',\n    \'regulatory_failure\': \'Unable or likely unable to meet regulatory requirements (Art. 2(c))\',\n    \'material_client_loss\': \'Likely loss of clients/counterparts materially affecting business (Art. 2(d))\',\n    \'data_adverse\': \'Data availability/authenticity/integrity/confidentiality impact adversely affects business objectives or regulatory obligations (Art. 9(5)(a))\',\n}\nNUM_FIELDS = {\n    \'clients\': \'Affected clients\', \'total_clients\': \'All clients using affected service\',\n    \'counterparts\': \'Affected financial counterparts\', \'total_counterparts\': \'All financial counterparts related to affected service\',\n    \'transactions\': \'Affected transaction count\', \'daily_transactions\': \'Daily average transaction count for affected service\',\n    \'transaction_value\': \'Affected transaction value (EUR)\', \'daily_value\': \'Daily average transaction value for affected service (EUR)\',\n    \'duration_hours\': \'Incident duration (hours; Art. 3)\',\n    \'downtime_minutes\': \'Downtime of ICT supporting critical/important functions (minutes)\',\n    \'member_states\': \'Number of EU Member States impacted (Art. 4)\',\n    \'economic_loss\': \'Incurred or likely total costs/losses (EUR; Art. 7, before recoveries)\',\n}\nCOUNT_FIELDS = {\'clients\',\'total_clients\',\'counterparts\',\'total_counterparts\',\'transactions\',\'member_states\'}\n\ndef tri_or(*values):\n    return True if True in values else (None if None in values else False)\n\ndef tri_and(*values):\n    return False if False in values else (None if None in values else True)\n\ndef gt(value, threshold):\n    return None if value is None else Decimal(str(value)) > Decimal(str(threshold))\n\ndef ratio(affected, total, threshold):\n    if affected is None or total is None or total == 0:\n        return None\n    return Decimal(str(affected)) > Decimal(str(total)) * Decimal(str(threshold))\n\ndef assess(values):\n    v = {k: values.get(k) for k in list(BOOL_FIELDS) + list(NUM_FIELDS)}\n    for k in BOOL_FIELDS:\n        if v[k] is not None and type(v[k]) is not bool:\n            raise ValueError(f\'{k}: use True, False or None\')\n    for k in NUM_FIELDS:\n        if v[k] is not None:\n            if isinstance(v[k], bool) or not isinstance(v[k], (float,int)) or not math.isfinite(v[k]) or v[k] < 0:\n                raise ValueError(f\'{k}: expected a finite non-negative number or unknown\')\n            if k in COUNT_FIELDS and v[k] != int(v[k]):\n                raise ValueError(f\'{k}: expected a whole number\')\n    if v[\'member_states\'] is not None and v[\'member_states\'] > 27:\n        raise ValueError(\'member_states cannot exceed 27\')\n    for a,b in [(\'clients\',\'total_clients\'),(\'counterparts\',\'total_counterparts\')]:\n        if v[a] is not None and v[b] is not None and v[a] > v[b]:\n            raise ValueError(f\'{a} cannot exceed {b}\')\n    gate = tri_or(v[\'critical_function\'],v[\'regulated_service\'],v[\'malicious_access\'])\n    # Art. 9(5)(b) expressly excludes access covered by Art. 9(5)(a).\n    access_b = tri_and(v[\'malicious_access\'],v[\'access_may_cause_loss\'],\n                       None if v[\'data_adverse\'] is None else not v[\'data_adverse\'])\n    criteria = {\n        \'Clients, counterparts and transactions — Art. 9(1)\': tri_or(\n            gt(v[\'clients\'],100000),ratio(v[\'clients\'],v[\'total_clients\'],.1),\n            ratio(v[\'counterparts\'],v[\'total_counterparts\'],.3),\n            ratio(v[\'transactions\'],v[\'daily_transactions\'],.1),\n            ratio(v[\'transaction_value\'],v[\'daily_value\'],.1),v[\'relevant_clients\']),\n        \'Reputation — Art. 9(2)\': tri_or(v[\'media\'],v[\'complaints\'],v[\'regulatory_failure\'],v[\'material_client_loss\']),\n        \'Duration / downtime — Art. 9(3)\': tri_or(gt(v[\'duration_hours\'],24),tri_and(v[\'critical_function\'],gt(v[\'downtime_minutes\'],120))),\n        \'Geographical spread — Art. 9(4)\': None if v[\'member_states\'] is None else v[\'member_states\'] >= 2,\n        \'Data losses — Art. 9(5)(a)\': v[\'data_adverse\'],\n        \'Economic impact — Art. 9(6)\': gt(v[\'economic_loss\'],100000),\n    }\n    met = sum(x is True for x in criteria.values())\n    unknown = sum(x is None for x in criteria.values())\n    two = True if met >= 2 else (None if met + unknown >= 2 else False)\n    major = tri_and(gate,tri_or(access_b,two))\n    return {\'status\': \'Major indicated\' if major is True else (\'Needs review\' if major is None else \'Single-incident major criteria not met\'),\n            \'major\':major,\'criticality_gate\':gate,\'access_9_5_b\':access_b,\'criteria\':criteria,\n            \'met_count\':met,\'unknown_count\':unknown}\n\ndef deadlines(awareness=None, classification=None, initial=None, intermediate=None):\n    """Unadjusted Art. 5 deadlines, UTC; no holiday/entity extension assumed."""\n    parsed = {}\n    for name,value in [(\'awareness\',awareness),(\'classification\',classification),(\'initial\',initial),(\'intermediate\',intermediate)]:\n        if value is None or value == \'\':\n            parsed[name] = None\n            continue\n        t = pd.Timestamp(value)\n        if t.tzinfo is None:\n            raise ValueError(f\'{name}: timezone offset required\')\n        parsed[name] = t.tz_convert(\'UTC\')\n    a,c,i,m = [parsed[k] for k in (\'awareness\',\'classification\',\'initial\',\'intermediate\')]\n    for earlier,later,label in [(a,c,\'classification before awareness\'),(c,i,\'initial before classification\'),(i,m,\'intermediate before initial\')]:\n        if earlier is not None and later is not None and later < earlier:\n            raise ValueError(label)\n    due = None\n    if a is not None and c is not None:\n        due = c + timedelta(hours=4)\n        if c <= a + timedelta(hours=24):\n            due = min(due,a + timedelta(hours=24))\n    return {\'initial_due_utc\':due,\n            \'intermediate_due_utc\':i + timedelta(hours=72) if i is not None else None,\n            \'final_due_utc\':m + pd.DateOffset(months=1) if m is not None else None}\n', encoding="utf-8")
Path('data/dora_scenarios.json').parent.mkdir(parents=True, exist_ok=True)
Path('data/dora_scenarios.json').write_text('[\n  {\n    "scenario_id": "SCN-01",\n    "name": "Core Retail Payment System Outage",\n    "system_affected": "Instant SEPA Payment Engine",\n    "description": "Database lock failure during high-volume processing leads to transaction timeouts for mobile banking users.",\n    "critical_service_affected": "Yes",\n    "downtime_minutes": 145,\n    "clients_affected": 120000,\n    "transactions_impacted": 45000,\n    "unauthorized_access": "No",\n    "geographic_impact": "Cross-border (DE, FR, NL)",\n    "estimated_economic_loss_eur": 850000,\n    "critical_function": true,\n    "malicious_access": false,\n    "clients": 120000,\n    "transactions": 45000,\n    "economic_loss": 850000,\n    "data_adverse": null,\n    "access_may_cause_loss": false,\n    "member_states": 3,\n    "assumptions": "Fictional scenario. Existing critical-service flag is interpreted as ICT supporting a critical/important function. Other unprovided facts remain unknown. Global impact does not imply two EU Member States."\n  },\n  {\n    "scenario_id": "SCN-02",\n    "name": "Tier-1 Cloud Provider Availability Disruption",\n    "system_affected": "Sovereign Credit Risk Calculators",\n    "description": "Regional network routing configuration error on hosting provider disrupts portfolio risk modeling tools.",\n    "critical_service_affected": "No",\n    "downtime_minutes": 310,\n    "clients_affected": 450,\n    "transactions_impacted": 0,\n    "unauthorized_access": "No",\n    "geographic_impact": "Single Member State (IE)",\n    "estimated_economic_loss_eur": 20000,\n    "critical_function": false,\n    "malicious_access": false,\n    "clients": 450,\n    "transactions": 0,\n    "economic_loss": 20000,\n    "data_adverse": null,\n    "access_may_cause_loss": false,\n    "member_states": 1,\n    "assumptions": "Fictional scenario. Existing critical-service flag is interpreted as ICT supporting a critical/important function. Other unprovided facts remain unknown. Global impact does not imply two EU Member States."\n  },\n  {\n    "scenario_id": "SCN-03",\n    "name": "Unauthorized High-Net-Worth Client Directory Access",\n    "system_affected": "Wealth Management CRM Portal",\n    "description": "API vulnerability exploited, exposing profiles and contact lists of premium wealth customers to an external IP.",\n    "critical_service_affected": "Yes",\n    "downtime_minutes": 0,\n    "clients_affected": 1200,\n    "transactions_impacted": 0,\n    "unauthorized_access": "Yes",\n    "geographic_impact": "Global (EU & US)",\n    "estimated_economic_loss_eur": 350000,\n    "critical_function": true,\n    "malicious_access": true,\n    "clients": 1200,\n    "transactions": 0,\n    "economic_loss": 350000,\n    "data_adverse": true,\n    "access_may_cause_loss": true,\n    "member_states": null,\n    "assumptions": "Fictional scenario. Existing critical-service flag is interpreted as ICT supporting a critical/important function. Other unprovided facts remain unknown. Global impact does not imply two EU Member States."\n  },\n  {\n    "scenario_id": "SCN-04",\n    "name": "Customer Facing Authentication Service Outage",\n    "system_affected": "Multi-Factor Authentication (MFA) Server",\n    "description": "Certificate expiration on authentication gateways prevents client logins to web-trading dashboards.",\n    "critical_service_affected": "Yes",\n    "downtime_minutes": 85,\n    "clients_affected": 65000,\n    "transactions_impacted": 11000,\n    "unauthorized_access": "No",\n    "geographic_impact": "Single Member State (IT)",\n    "estimated_economic_loss_eur": 120000,\n    "critical_function": true,\n    "malicious_access": false,\n    "clients": 65000,\n    "transactions": 11000,\n    "economic_loss": 120000,\n    "data_adverse": null,\n    "access_may_cause_loss": false,\n    "member_states": 1,\n    "assumptions": "Fictional scenario. Existing critical-service flag is interpreted as ICT supporting a critical/important function. Other unprovided facts remain unknown. Global impact does not imply two EU Member States."\n  },\n  {\n    "scenario_id": "SCN-05",\n    "name": "SWIFT Connectivity Gateway Timeout",\n    "system_affected": "SWIFT Alliance Gateway",\n    "description": "Corrupted message payload hangs queuing thread, delaying end-of-day international treasury settlements.",\n    "critical_service_affected": "Yes",\n    "downtime_minutes": 190,\n    "clients_affected": 85,\n    "transactions_impacted": 240,\n    "unauthorized_access": "No",\n    "geographic_impact": "Cross-border (Global)",\n    "estimated_economic_loss_eur": 1950000,\n    "critical_function": true,\n    "malicious_access": false,\n    "clients": 85,\n    "transactions": 240,\n    "economic_loss": 1950000,\n    "data_adverse": null,\n    "access_may_cause_loss": false,\n    "member_states": null,\n    "assumptions": "Fictional scenario. Existing critical-service flag is interpreted as ICT supporting a critical/important function. Other unprovided facts remain unknown. Global impact does not imply two EU Member States."\n  }\n]\n', encoding="utf-8")
Path('data/dora_scenarios.csv').parent.mkdir(parents=True, exist_ok=True)
Path('data/dora_scenarios.csv').write_text('scenario_id,name,system_affected,description,critical_service_affected,downtime_minutes,clients_affected,transactions_impacted,unauthorized_access,geographic_impact,estimated_economic_loss_eur\nSCN-01,Core Retail Payment System Outage,Instant SEPA Payment Engine,Database lock failure during high-volume processing leads to transaction timeouts for mobile banking users.,Yes,145,120000,45000,No,"Cross-border (DE, FR, NL)",850000\nSCN-02,Tier-1 Cloud Provider Availability Disruption,Sovereign Credit Risk Calculators,Regional network routing configuration error on hosting provider disrupts portfolio risk modeling tools.,No,310,450,0,No,Single Member State (IE),20000\nSCN-03,Unauthorized High-Net-Worth Client Directory Access,Wealth Management CRM Portal,"API vulnerability exploited, exposing profiles and contact lists of premium wealth customers to an external IP.",Yes,0,1200,0,Yes,Global (EU & US),350000\nSCN-04,Customer Facing Authentication Service Outage,Multi-Factor Authentication (MFA) Server,Certificate expiration on authentication gateways prevents client logins to web-trading dashboards.,Yes,85,65000,11000,No,Single Member State (IT),120000\nSCN-05,SWIFT Connectivity Gateway Timeout,SWIFT Alliance Gateway,"Corrupted message payload hangs queuing thread, delaying end-of-day international treasury settlements.",Yes,190,85,240,No,Cross-border (Global),1950000\n', encoding="utf-8")
from prepare_data import build
incidents_df, quality = build('incident_event_log.csv','data')
assert len(incidents_df) == df['number'].nunique()
print('Generated data/incidents.csv and data/data_quality.json')


### Step 2: Write Streamlit Application Code (`app.py`)
We will create a comprehensive, single-file Streamlit application featuring the Interactive Operations Dashboard and the DORA Assessment Screening Assistant.

In [ ]:
from pathlib import Path
Path('app.py').parent.mkdir(parents=True, exist_ok=True)
Path('app.py').write_text('import streamlit as st\nimport pandas as pd\nimport json\nfrom pathlib import Path\nfrom prepare_data import boolean_series\nfrom rules import assess, deadlines, BOOL_FIELDS, NUM_FIELDS, COUNT_FIELDS\nBASE = Path(__file__).resolve().parent\nimport plotly.express as px\n\nst.set_page_config(\n    page_title="DORA ICT Incident Triage & Monitor",\n    page_icon="🛡️",\n    layout="wide"\n)\n\n# Load processed datasets\n@st.cache_data\ndef load_data(base_path=str(BASE)):\n    base = Path(base_path)\n    incidents = pd.read_csv(base / \'data/incidents.csv\',low_memory=False)\n    scenarios = json.loads((base / \'data/dora_scenarios.json\').read_text())\n    for col in [\'opened_at\',\'resolved_at\']:\n        incidents[col] = pd.to_datetime(incidents[col], errors=\'coerce\')\n    incidents[\'made_sla\'] = boolean_series(incidents[\'made_sla\'])\n    if incidents[\'number\'].isna().any() or incidents[\'number\'].duplicated().any():\n        raise ValueError(\'Incident IDs must be present and unique\')\n    return incidents, scenarios\n\ntry:\n    df_incidents, df_scenarios = load_data(str(BASE))\nexcept Exception as e:\n    st.error(f"Error loading processed datasets: {e}. Please ensure data/incidents.csv and data/dora_scenarios.csv exist.")\n    st.stop()\n\nst.title("🛡️ DORA-Informed ICT Incident Triage & Reporting Monitor")\nst.markdown("""\nAn operational dashboard and educational screening environment mapped to **DORA Regulation (EU) 2022/2554**\nand adopted Delegated Regulations **2024/1772** (classification) and **2025/301** (reporting).\nEducational portfolio MVP; no claim of DORA compliance. No regulatory submission or persistent incident tracking.\n""")\n\ntab1, tab2, tab3 = st.tabs(["📊 Incident Operations Dashboard", "🔍 DORA Assessment Screening", "📘 Reference Guide"])\n\n# TAB 1: Incident Operations Dashboard\nwith tab1:\n    st.header("Incident Operations Overview")\n    st.markdown("Historical, anonymised public UCI ServiceNow dataset from an IT company; not a financial institution incident register.")\n\n    # Filtering sidebar options simulated in columns\n    col1, col2, col3 = st.columns(3)\n    with col1:\n        state_filter = st.multiselect("Latest Incident State", options=df_incidents[\'incident_state\'].dropna().unique().tolist(), default=None)\n    with col2:\n        priority_filter = st.multiselect("Incident Priority", options=df_incidents[\'priority\'].dropna().unique().tolist(), default=None)\n    with col3:\n        sla_filter = st.selectbox("SLA Made Status", ["All", "True Only", "False Only"])\n\n    # Apply filters\n    filtered_df = df_incidents.copy()\n    if state_filter:\n        filtered_df = filtered_df[filtered_df[\'incident_state\'].isin(state_filter)]\n    if priority_filter:\n        filtered_df = filtered_df[filtered_df[\'priority\'].isin(priority_filter)]\n    if sla_filter == "True Only":\n        filtered_df = filtered_df[filtered_df[\'made_sla\'] == True]\n    elif sla_filter == "False Only":\n        filtered_df = filtered_df[filtered_df[\'made_sla\'] == False]\n\n    # Key Performance Indicators\n    kp1, kp2, kp3, kp4 = st.columns(4)\n    kp1.metric("Total Unique Incidents", f"{len(filtered_df):,}")\n\n    closed_count = len(filtered_df[filtered_df[\'incident_state\'] == \'Closed\'])\n    kp2.metric("Closed Incidents", f"{closed_count:,}")\n\n    avg_res_time = filtered_df[\'resolution_duration_hours\'].mean()\n    kp3.metric("Avg. Resolution Duration", f"{avg_res_time:.1f} Hours" if not pd.isna(avg_res_time) else "N/A")\n\n    sla_known = filtered_df[\'made_sla\'].dropna()\n    sla_met_pct = sla_known.mean() * 100 if len(sla_known) else None\n    kp4.metric("SLA Met Rate (known outcomes)", f"{sla_met_pct:.1f}%" if sla_met_pct is not None else "N/A")\n    st.caption(f"Resolution mean uses {filtered_df[\'resolution_duration_hours\'].notna().sum():,} valid resolved/closed records. SLA unknown: {filtered_df[\'made_sla\'].isna().sum():,}. Neither metric establishes DORA compliance.")\n    if filtered_df.empty:\n        st.info("No incidents match these filters.")\n\n    # Visualizations\n    v1, v2 = st.columns(2)\n    with v1:\n        st.subheader("Incidents by Priority State")\n        p_counts = filtered_df[\'priority\'].value_counts().rename_axis(\'priority\').reset_index(name=\'count\')\n        fig_p = px.bar(p_counts, x=\'priority\', y=\'count\', color=\'priority\', labels={\'count\': \'Number of Incidents\'})\n        st.plotly_chart(fig_p, use_container_width=True)\n\n    with v2:\n        st.subheader("Incident Volumes Over Time (Opened Month)")\n        filtered_df[\'opened_month\'] = filtered_df[\'opened_at\'].dt.to_period(\'M\')\n        trend_df = filtered_df.groupby(\'opened_month\').size().reset_index(name=\'Incident Count\')\n        trend_df[\'opened_month\'] = trend_df[\'opened_month\'].astype(str)\n        fig_t = px.line(trend_df, x=\'opened_month\', y=\'Incident Count\', markers=True)\n        st.plotly_chart(fig_t, use_container_width=True)\n\n    # Table view\n    st.subheader("Incident Record Directory (First 100 Rows)")\n    st.dataframe(filtered_df[[\'number\', \'incident_state\', \'active\', \'priority\', \'impact\', \'urgency\', \'reassignment_count\', \'made_sla\', \'resolution_duration_hours\']].head(100), width=\'stretch\')\n\n# TAB 2: DORA Assessment Screening Simulator\nwith tab2:\n    st.header("DORA Major Incident Screening Tool")\n    st.info("Educational single-incident screening. Unknown facts remain unknown. Review the evidence with the responsible incident/risk team; this app makes no official classification.")\n    selected = st.selectbox("Load a simulated ICT scenario", [\'Custom input\'] + [s[\'name\'] for s in df_scenarios])\n    scenario = next((s for s in df_scenarios if s[\'name\'] == selected), {})\n    if scenario:\n        st.write(scenario[\'system_affected\'])\n        st.write(scenario[\'description\'])\n        st.caption(scenario[\'assumptions\'])\n    st.caption("Choose Unknown for unestablished facts. Zero means a confirmed zero. Percent denominators must refer to the affected service; daily transaction averages follow Art. 1(2).")\n    values = {}\n    f1, f2 = st.columns(2)\n    for index, (key,label) in enumerate(BOOL_FIELDS.items()):\n        with [f1,f2][index % 2]:\n            initial = scenario.get(key)\n            options = [\'Unknown\',\'Yes\',\'No\']\n            chosen = st.selectbox(label,options,index=0 if initial is None else (1 if initial else 2),key=f\'{selected}_{key}\')\n            values[key] = {\'Unknown\':None,\'Yes\':True,\'No\':False}[chosen]\n    st.caption("Data impact in Art. 9(5)(a) includes temporary inaccessibility with adverse business/regulatory consequences. Art. 9(5)(b) is access not covered by (a), with possible data loss; a suspected breach alone is insufficient.")\n    for index, (key,label) in enumerate(NUM_FIELDS.items()):\n        with [f1,f2][index % 2]:\n            initial = scenario.get(key)\n            count = key in COUNT_FIELDS\n            values[key] = st.number_input(label,min_value=0 if count else 0.0,\n                value=None if initial is None else (int(initial) if count else float(initial)),\n                step=1 if count else 0.01,key=f\'{selected}_{key}\',placeholder=\'Unknown\')\n    st.caption("Geography counts EU Member States impacted, considering Art. 4 significance; worldwide scope alone is insufficient. Economic loss includes Art. 7 direct/indirect incident costs before recoveries, excluding normal operating/improvement costs.")\n    try:\n        result = assess(values)\n    except ValueError as e:\n        st.error(f"Invalid screening input: {e}")\n    else:\n        {\'Major indicated\':st.error,\'Needs review\':st.warning,\'Single-incident major criteria not met\':st.info}[result[\'status\']](result[\'status\'])\n        label = lambda v: \'Met\' if v is True else (\'Unknown\' if v is None else \'Not met\')\n        st.write(f"Criticality gate (Art. 6): {label(result[\'criticality_gate\'])}. Special access trigger (Art. 9(5)(b)): {label(result[\'access_9_5_b\'])}.")\n        st.dataframe(pd.DataFrame([{\'Criterion\':k,\'Result\':label(v)} for k,v in result[\'criteria\'].items()]),hide_index=True,width=\'stretch\')\n        st.caption(f"{result[\'met_count\']} of six ordinary categories met; {result[\'unknown_count\']} unknown. Art. 8 requires the criticality gate plus the special access trigger OR at least two ordinary categories. Multiple client/transaction triggers still count as one category.")\n    st.warning("Recurring incidents are not automatically aggregated. Art. 8(2): assess monthly; at least twice in six months, same apparent root cause, collectively meeting major criteria. Exemptions apply to microenterprises and Art. 16(1) entities. A negative single-incident result does not clear recurring incidents.")\n    st.subheader("Reporting deadline helper")\n    st.markdown("Initial: as early as possible, within **4 hours after major classification**, normally capped at **24 hours after awareness**. If classified after 24 hours, within 4 hours of that later classification. Intermediate: **72 hours after initial submission**; update without undue delay and when regular activities recover. Final: **one calendar month after the latest intermediate submission**.")\n    st.caption("Unadjusted deadlines only. Art. 5(4) permits certain weekend/bank-holiday deadlines to move to noon next working day. Initial/intermediate exceptions and competent-authority exclusions apply under Art. 5(5)-(6). No extension is assumed here; verify entity type, local calendar and authority decisions. If unable to report on time, notify the authority by the deadline with reasons.")\n    with st.expander("Calculate unadjusted deadlines (explicit timezone required)"):\n        times = {}\n        for key, text in [(\'awareness\',\'Awareness\'),(\'classification\',\'Major classification\'),(\'initial\',\'Actual initial submission\'),(\'intermediate\',\'Latest actual intermediate submission\')]:\n            times[key] = st.text_input(text + \' — ISO 8601 with timezone offset\',placeholder=\'2026-11-19T12:00:00+01:00\',key=\'time_\'+key)\n        try:\n            due = deadlines(**times)\n            for name, value in due.items():\n                st.write(name.replace(\'_\',\' \')+\': \'+(value.isoformat() if value is not None else \'Awaiting required timestamps\'))\n        except (ValueError,TypeError) as e:\n            st.error(f\'Invalid timestamp: {e}\')\n    st.caption("Keep an internal log and escalate indicated major incidents promptly. Client communications under DORA Art. 19(3) depend on impact on clients\' financial interests; notifications are not sent by this app.")\n\nwith tab3:\n    st.header("Regulatory Reference Framework")\n    st.markdown("""\n- [DORA — Regulation (EU) 2022/2554, Arts. 17–20](https://eur-lex.europa.eu/eli/reg/2022/2554/oj/eng)\n- [Delegated Regulation (EU) 2024/1772, Arts. 1–9](https://eur-lex.europa.eu/eli/reg_del/2024/1772/oj/eng): classification criteria, six materiality categories and criticality gate.\n- [Delegated Regulation (EU) 2025/301, Art. 5](https://eur-lex.europa.eu/eli/reg_del/2025/301/oj/eng): reporting clocks, later classification and holiday exceptions.\n- [Implementing Regulation (EU) 2025/302](https://eur-lex.europa.eu/eli/reg_impl/2025/302/oj/eng): official reporting forms/procedures; this MVP does not implement or submit them.\n\n**Limitations:** No automated recurring-incident aggregation, authority submission, alert scheduling, persistence, entity-scope determination, or bank-holiday extension calculation. The historical ticket dataset lacks DORA client denominators, downtime, criticality evidence and reporting timestamps; it is used only for operational analysis. Fictional scenarios are kept separate. Inputs are not saved by this app.\n\n**Data attribution:** Amaral, C., Fantinato, M., & Peres, S. (2018), Incident management process enriched event log, UCI Machine Learning Repository, [doi:10.24432/C57S4H](https://doi.org/10.24432/C57S4H), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Anonymised public data from an IT company; not the user\'s organisation. Preprocessing selects whole latest snapshots by numeric revision count, then update timestamp and original row order; ties are flagged. Unknown outcomes and invalid resolution durations are excluded from the relevant means.\n""")\n    with st.expander("Dataset quality audit"):\n        st.json(json.loads((BASE / \'data/data_quality.json\').read_text()))\n', encoding="utf-8")
print('Updated app.py')


### Step 3: Write Metadata and Dependency Files (`requirements.txt`, `README.md`)
We will write exact configurations for immediate deployment to Streamlit Community Cloud.

In [ ]:
from pathlib import Path
Path('requirements.txt').parent.mkdir(parents=True, exist_ok=True)
Path('requirements.txt').write_text('streamlit==1.50.0\npandas==2.3.3\nplotly==7.1.0\n', encoding="utf-8")
Path('requirements-dev.txt').parent.mkdir(parents=True, exist_ok=True)
Path('requirements-dev.txt').write_text('-r requirements.txt\npytest==8.4.2\n', encoding="utf-8")
Path('README.md').parent.mkdir(parents=True, exist_ok=True)
Path('README.md').write_text("# DORA-Informed ICT Incident Monitor\n\nEducational portfolio MVP with three tabs: historical operations dashboard, single-incident screening and reference guide. **Not a claim or certification of DORA compliance.** No authority submissions, live monitoring, saved case management or scheduled alerts.\n\n## Run\n\nUse Python 3.12. From the repository root:\n\n```sh\npip install -r requirements.txt\nstreamlit run app.py\n```\n\nThe prepared dataset is included. No Google Drive, Colab, external API keys or secrets are required at runtime.\n\n## Classification\n\nRules are in `rules.py`, separated from the existing Streamlit dashboard so they can be tested. Delegated Regulation (EU) 2024/1772, Article 8(1), requires the Article 6 criticality gate **and** either Article 9(5)(b), or at least two other Article 9 materiality categories. Criticality is a gate, not one of the two categories.\n\n| Category | Implemented threshold |\n|---|---|\n| Clients / counterparts / transactions, Art. 9(1) | Any: affected clients >10% of service clients or >100,000; counterparts >30%; transaction count or value >10% of daily average for affected service; relevant clients/counterparts affected. All count as one category. |\n| Reputation, Art. 9(2) | Any of the four Article 2 conditions. |\n| Duration / downtime, Art. 9(3) | Incident >24h, or downtime >2h for ICT supporting critical/important functions. One category. |\n| Geography, Art. 9(4) | Impact in at least two EU Member States, assessed under Article 4. |\n| Data, Art. 9(5)(a) | Adverse business/regulatory effect of availability, authenticity, integrity or confidentiality loss. |\n| Economic, Art. 9(6) | Costs/losses exceeded or likely to exceed EUR 100,000, under Article 7 before recoveries. |\n\nArticle 9(5)(b) special path requires successful, malicious AND unauthorised system access, not covered by (a), that may cause data losses. A suspicion alone does not meet it. The gate can be met through ICT supporting critical/important functions, a regulated financial service, or successful malicious unauthorised access (Article 6(a)-(c)).\n\nUnknowns are not filled with zeros or False. The result is `Needs review` where missing evidence could change the conclusion. Zero denominator produces an unknown ratio, not division by zero. Invalid, negative, non-finite or inconsistent inputs are rejected. Counts and EU Member State counts are validated.\n\n## Reporting clocks\n\nDelegated Regulation (EU) 2025/301, Article 5:\n\n- Initial notification as early as possible, within four hours of major classification, normally no later than 24 hours from awareness. Later classification after those 24 hours permits four hours from later classification.\n- Intermediate within 72 hours of actual initial submission. Updates without undue delay and when regular activities recover.\n- Final within one calendar month of actual latest intermediate submission, not 30 days from incident detection.\n- Notify the competent authority by the applicable deadline, with reasons, if reporting on time is impossible.\n\nThe helper calculates unadjusted UTC deadlines with explicit timezone offsets and validates chronology. It does **not** apply Article 5(4)-(6) weekend/bank-holiday extensions or entity/authority exceptions. Calendar months are calculated in UTC; confirm national/local time conventions and DST treatment for operational use. Missing clock origins leave the corresponding deadline unknown.\n\n## Data preparation and provenance\n\nDataset: Amaral, C., Fantinato, M., & Peres, S. (2018), *Incident management process enriched event log*. UCI Machine Learning Repository. https://doi.org/10.24432/C57S4H. License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). This project adapts the public data through cleaning and incident-level snapshot selection. The data came from an IT company, not a financial institution or the user's organisation.\n\nThe source file SHA-256 was read in the live Colab runtime and independently matched to UCI: `fd184bbfd62329cfe093e99da2ea7071905f2ead91900b448eb2635870821bef`.\n\n`prepare_data.py` normalises `?`, blanks and nulls; parses source dates with explicit day-first format; sorts numeric revision counts, then update time and stable original row order; selects a whole latest snapshot; and flags ties. It preserves unknown fields instead of splicing older non-null values into a new snapshot. Missing identifiers or revision order are rejected. Resolution duration is calculated only for resolved/closed records with valid non-negative opened-to-resolved intervals. It is **not** DORA incident duration or downtime. SLA rate excludes unknown outcomes and shows its denominator/unknown count. Empty filters show N/A.\n\n```sh\npython prepare_data.py /path/to/incident_event_log.csv --output-dir data\npip install -r requirements-dev.txt\npython -m pytest -q\n```\n\nThe public event data has 141,712 records / 36 columns and 24,918 incident IDs. The generated incident data has one row per ID. All latest records in this historical sample are Closed; that is not a claim about live operations. There are 1,556 missing usable resolution durations. Detailed values are in `data/data_quality.json`.\n\nThe original five fictional scenarios are retained. JSON supplies the screening fields; the original CSV is kept for reference. Additional unprovided facts remain unknown. The original critical-service flag is explicitly interpreted as ICT supporting a critical/important function for these fictional examples. Global impact does not establish an EU Member State count. SCN-01/03/05 indicate major; SCN-02/04 require review with the available facts.\n\n## Scope limits\n\n- Recurring-incident aggregation (Art. 8(2)) is **manual**: at least twice in six months, same apparent root cause and collectively major criteria, assessed monthly, subject to microenterprise/Art. 16(1) exclusions. A negative individual result does not clear recurrence.\n- Financial entity applicability, evidential judgements, reference-period estimates, regulatory scope and competent authority are user responsibilities.\n- Dataset contains no substantiated regulatory client denominators, downtime, criticality, malicious-access assessments or reporting timestamps. Historical tickets are not automatically classified.\n- No regulatory forms, submission, significant-cyber-threat classifier, persistence, access control, production incident handling or notification service.\n- User inputs are not explicitly persisted by this code. A public Community Cloud app is intended for fictional examples; no confidentiality assessment of the hosting platform was performed.\n\n## Deploy to Streamlit Community Cloud\n\n1. Put these files at the repository root with the `data/` folder intact.\n2. At https://share.streamlit.io choose Create app, select the GitHub repository and `main` branch, entry point `app.py`.\n3. Set Python **3.12** in Advanced settings. No secrets are needed.\n4. Deploy; verify dashboard rendering, a major scenario, a needs-review scenario, an empty filter and the reporting helper.\n\n## Official references\n\n- [Regulation (EU) 2022/2554](https://eur-lex.europa.eu/eli/reg/2022/2554/oj/eng), Arts. 17–20.\n- [Delegated Regulation (EU) 2024/1772](https://eur-lex.europa.eu/eli/reg_del/2024/1772/oj/eng), Arts. 1–9.\n- [Delegated Regulation (EU) 2025/301](https://eur-lex.europa.eu/eli/reg_del/2025/301/oj/eng), Art. 5.\n- [Implementing Regulation (EU) 2025/302](https://eur-lex.europa.eu/eli/reg_impl/2025/302/oj/eng), official forms/procedures, outside MVP implementation.\n", encoding="utf-8")
Path('CHANGELOG_AND_VALIDATION.md').parent.mkdir(parents=True, exist_ok=True)
Path('CHANGELOG_AND_VALIDATION.md').write_text("# Changes and validation — 19 November 2026\n\n## Changes\n\n- Preserved the operations dashboard, scenario selector, interactive screening and reference tabs.\n- Replaced incorrect 50,000-client threshold with >100,000 / >10% alternatives; added counterparts, transaction count/value and relevant-client alternatives, grouped as one category.\n- Implemented the Article 6 criticality gate separately; ordinary classification requires two categories. Added reputation, geographic impact in EU Member States, duration and adverse data effects. Implemented the qualified Art. 9(5)(b) access path, excluding cases covered by (a).\n- Unknown values remain unknown; added Needs review results, ratio-denominator validation and rejection of invalid numbers. Global geography is no longer silently reset to single-state impact or assumed cross-EU.\n- Replaced the incorrect four-hours-from-detection statement; implemented awareness/classification clocks, late classification, 72-hour intermediate and calendar-month final deadline origins. Documented holiday/entity/authority exceptions; no extension is assumed.\n- Kept the five fictional scenarios with explicit interpretations and missing facts; did not manufacture client/transaction denominators or EU geography.\n- Normalised ?, blanks and nulls; numeric revision sorting and stable whole-row snapshot selection; ties flagged, missing identifiers/revisions rejected. Replaced exploratory groupby.last() snapshot splicing.\n- Kept 24,918 incidents. On this source the revised snapshot ordering selects the same final revision as the previous timestamp-first ordering (zero differences).\n- Restricted resolution averages to valid resolved/closed observations. Missing durations remain missing; not inferred from closure time. SLA rates use known boolean outcomes; empty subsets show N/A. No regulatory metrics are inferred from ticket resolution or SLA.\n- Corrected dataset attribution: public anonymised UCI IT-company data, not the user's organisation or a bank. Added CC BY 4.0 attribution.\n- Pinned runtime dependencies, made paths independent of working directory, made cache keys include dataset directory, and regenerated app/data/notebook packaging. Bundle generation fails if required files are absent.\n- Removed references to adopted delegated regulations as Joint Draft RTS. Added official sources and explicit MVP limitations.\n\n## Executed validation\n\nPython 3.12.13, Streamlit 1.50.0, pandas 2.3.3, Plotly 7.1.0, pytest 8.4.2. Command: `python -m pytest -q`. **37 tests passed**, no pytest warnings (see `test_results.xml`).\n\nCoverage includes each material threshold at and above the boundary; grouped category counting; criticality gate alternatives; special access versus adverse-data path; unknowns; zero denominators; negative/NaN/infinite values; inconsistent affected-client counts; all five scenarios; early/late/exact-24h reporting clocks; 72h origin; leap-year/month-end final deadlines; explicit timezone and chronology validation; whole-row selection; numeric 9-to-10 revision ordering with conflicting timestamps; placeholder/blank/boolean handling; missing/invalid/negative resolution times; tied snapshots; latest-ID integrity; Streamlit initial rendering, all scenarios, resets, SLA filters, empty results, invalid screening inputs and reporting-helper interactions.\n\n### Data evidence\n\nThe raw CSV hash was read in the existing live Colab runtime and matches the independently downloaded official UCI file: `fd184bbfd62329cfe093e99da2ea7071905f2ead91900b448eb2635870821bef`.\n\n| Check | Result |\n|---|---:|\n| Source records / columns | 141,712 / 36 |\n| Unique incident IDs / generated rows | 24,918 / 24,918 |\n| Exact duplicate source rows | 0 |\n| Invalid non-missing parsed operational timestamps | 0 |\n| Negative resolution durations | 0 |\n| Missing usable incident resolution durations | 1,556 |\n| Valid resolution observations | 23,362 |\n| Latest snapshot ties | 0 |\n| Unknown latest SLA outcomes | 0 |\n| Latest state Closed | 24,918 |\n| Mean / median valid resolution hours | 178.171582 / 22.1 |\n\nOriginal notebook app.py, scenarios and processed-file existence were inspected in the runtime. Original app.py SHA-256: `de9c4e0a731176bbbe9ade206be34503d777cafe2d8998de8bc13c1091d3115c`. Changes were based on that inspected app and the notebook generator, not a replacement design.\n\n## Unverified / excluded\n\n- No claim of DORA compliance or external legal validation. Entity applicability, factual judgements and competent-authority reporting decisions are not verified.\n- No automatic aggregation of recurring incidents or significant-cyber-threat classifier.\n- Holiday calendars, entity exceptions, authority overrides and local/DST treatment are not automated; displayed deadlines are unadjusted UTC calculations.\n- No forms/submissions under 2025/302, client notification, persistence, real-time feed, scheduler or access-control testing.\n- Local AppTest execution is not proof of a public deployment. Cloud build, Linux runtime and anonymous public access must be verified separately after deployment; no public URL is claimed by this report.\n- Full exploratory Colab Run all has not been rerun during local validation. The rewritten generator and tests are self-contained for the runtime, but cloud/Colab execution is a separate check.\n\nOfficial legal sources and implementation scope are linked in README.md and the app's Reference Guide.\n", encoding="utf-8")
Path('pytest.ini').parent.mkdir(parents=True, exist_ok=True)
Path('pytest.ini').write_text('[pytest]\npythonpath = .\ntestpaths = tests\n', encoding="utf-8")
Path('.gitignore').parent.mkdir(parents=True, exist_ok=True)
Path('.gitignore').write_text('__pycache__/\n.pytest_cache/\n.venv/\n*.zip\n.streamlit/secrets.toml\n', encoding="utf-8")
Path('tests/test_app.py').parent.mkdir(parents=True, exist_ok=True)
Path('tests/test_app.py').write_text("from pathlib import Path\nfrom streamlit.testing.v1 import AppTest\nAPP=Path(__file__).resolve().parents[1]/'app.py'\n\ndef find(elements,label):\n    return next(e for e in elements if e.label==label)\n\ndef test_default_and_all_scenarios():\n    at=AppTest.from_file(str(APP)).run(timeout=60)\n    assert not at.exception\n    assert at.metric[0].value=='24,918'\n    scenario=find(at.selectbox,'Load a simulated ICT scenario')\n    for name in scenario.options[1:]:\n        find(at.selectbox,'Load a simulated ICT scenario').select(name).run(timeout=30)\n        assert not at.exception\n    find(at.selectbox,'Load a simulated ICT scenario').select('Core Retail Payment System Outage').run(timeout=30)\n    assert at.error[0].value=='Major indicated'\n    find(at.selectbox,'Load a simulated ICT scenario').select('Customer Facing Authentication Service Outage').run(timeout=30)\n    assert any(w.value=='Needs review' for w in at.warning)\n\ndef test_empty_filter_and_sla():\n    at=AppTest.from_file(str(APP)).run(timeout=60)\n    find(at.selectbox,'SLA Made Status').select('False Only').run(timeout=30)\n    assert not at.exception and at.metric[3].value=='0.0%'\n    # Pick a priority then an SLA combination with no results if any available.\n    import pandas as pd\n    from prepare_data import boolean_series\n    d=pd.read_csv(APP.parent/'data/incidents.csv',low_memory=False);d.made_sla=boolean_series(d.made_sla)\n    for p in d.priority.unique():\n        if not ((d.priority==p)&(d.made_sla==False)).any():\n            find(at.multiselect,'Incident Priority').set_value([p]).run(timeout=30)\n            assert not at.exception and at.metric[0].value=='0' and at.metric[3].value=='N/A'\n            break\n\ndef test_deadline_inputs():\n    at=AppTest.from_file(str(APP)).run(timeout=60)\n    at.text_input[0].set_value('2026-11-19T00:00:00Z')\n    at.text_input[1].set_value('2026-11-19T01:00:00Z').run(timeout=30)\n    assert not at.exception\n    assert any('2026-11-19T05:00:00+00:00' in x.value for x in at.markdown)\n    at.text_input[1].set_value('invalid').run(timeout=30)\n    assert not at.exception and any('Invalid timestamp' in e.value for e in at.error)\n\ndef test_empty_dataset_filter_path(tmp_path):\n    import shutil\n    import pandas as pd\n    for f in ['app.py','rules.py','prepare_data.py']:\n        shutil.copy(APP.parent/f,tmp_path/f)\n    shutil.copytree(APP.parent/'data',tmp_path/'data')\n    data=pd.read_csv(tmp_path/'data/incidents.csv',low_memory=False).head(2).copy()\n    data['incident_state']=['Closed','Resolved'];data['made_sla']=[True,False]\n    data.to_csv(tmp_path/'data/incidents.csv',index=False)\n    at=AppTest.from_file(str(tmp_path/'app.py')).run(timeout=60)\n    find(at.multiselect,'Latest Incident State').set_value(['Closed'])\n    find(at.selectbox,'SLA Made Status').select('False Only').run(timeout=30)\n    assert not at.exception and at.metric[0].value=='0'\n    assert at.metric[2].value=='N/A' and at.metric[3].value=='N/A'\n    assert any('No incidents match' in x.value for x in at.info)\n\ndef test_scenario_reset_and_invalid_input():\n    from rules import NUM_FIELDS\n    at=AppTest.from_file(str(APP)).run(timeout=60)\n    find(at.selectbox,'Load a simulated ICT scenario').select('Core Retail Payment System Outage').run(timeout=30)\n    assert find(at.number_input,NUM_FIELDS['member_states']).value==3\n    find(at.selectbox,'Load a simulated ICT scenario').select('Customer Facing Authentication Service Outage').run(timeout=30)\n    assert find(at.number_input,NUM_FIELDS['member_states']).value==1\n    find(at.number_input,NUM_FIELDS['member_states']).set_value(28).run(timeout=30)\n    assert not at.exception and any('Invalid screening input' in x.value for x in at.error)\n", encoding="utf-8")
Path('tests/test_data.py').parent.mkdir(parents=True, exist_ok=True)
Path('tests/test_data.py').write_text("from pathlib import Path\nimport json\nimport pandas as pd\nimport pytest\nfrom prepare_data import prepare,boolean_series\nROOT=Path(__file__).resolve().parents[1]\n\ndef row(**updates):\n    r=dict(number='INC1',incident_state='Closed',sys_mod_count='1',made_sla='False',active='False',priority='3 - Moderate',impact='2 - Medium',urgency='2 - Medium',reassignment_count='0',opened_at='01/01/2026 00:00',sys_updated_at='02/01/2026 00:00',resolved_at='01/01/2026 02:00',closed_at='02/01/2026 00:00')\n    r.update(updates);return r\n\ndef test_generated_integrity():\n    d=pd.read_csv(ROOT/'data/incidents.csv',low_memory=False)\n    q=json.loads((ROOT/'data/data_quality.json').read_text())\n    assert len(d)==24918 and d.number.nunique()==len(d) and d.number.notna().all()\n    assert q['event_rows']==141712 and q['source_columns']==36\n    assert q['raw_sha256']=='fd184bbfd62329cfe093e99da2ea7071905f2ead91900b448eb2635870821bef'\n    assert q['exact_duplicate_rows']==0\n    assert d.resolution_duration_hours.ge(0).sum()==23362\n    assert d.resolution_duration_hours.isna().sum()==1556\n    t=(pd.to_datetime(d.resolved_at)-pd.to_datetime(d.opened_at)).dt.total_seconds()/3600\n    pd.testing.assert_series_equal(d.resolution_duration_hours,t,check_names=False)\n    assert (d.incident_state=='Closed').all()\n\ndef test_revision_order_numeric_and_whole_snapshot():\n    raw=pd.DataFrame([row(sys_mod_count='9',priority='old',sys_updated_at='03/01/2026 00:00'),row(sys_mod_count='10',priority='?',sys_updated_at='02/01/2026 00:00')])\n    d,q=prepare(raw)\n    assert d.iloc[0].sys_mod_count==10 and pd.isna(d.iloc[0].priority)\n    assert not d.iloc[0].made_sla\n\ndef test_missing_invalid_and_negative():\n    d,q=prepare(pd.DataFrame([row(resolved_at='?',made_sla='?',priority='   '),row(number='INC2',resolved_at='31/12/2025 23:00'),row(number='INC3',resolved_at='invalid')]))\n    assert d.resolution_duration_hours.isna().all()\n    assert q['negative_durations']==1 and q['invalid_datetimes']['resolved_at']==1\n    assert q['unknown_sla']==1 and pd.isna(d.iloc[0].priority)\n\ndef test_unresolved_and_tied_snapshots():\n    d,q=prepare(pd.DataFrame([row(incident_state='Active'),row(incident_state='Active',priority='second')]))\n    assert pd.isna(d.iloc[0].resolution_duration_hours)\n    assert d.iloc[0].priority=='second' and d.iloc[0].snapshot_tie\n\n@pytest.mark.parametrize('change',[{'number':'?'},{'sys_mod_count':'?'},{'sys_mod_count':'oops'}])\ndef test_reject_invalid_identity_or_order(change):\n    with pytest.raises(ValueError):prepare(pd.DataFrame([row(**change)]))\n\ndef test_boolean_not_string_truthiness():\n    s=boolean_series(pd.Series(['False','True','?',None,'oops']))\n    assert s.iloc[0]==False and s.iloc[1]==True and s.isna().sum()==3\n", encoding="utf-8")
Path('tests/test_rules.py').parent.mkdir(parents=True, exist_ok=True)
Path('tests/test_rules.py').write_text("import json\nfrom pathlib import Path\nimport pytest\nfrom rules import assess,deadlines,BOOL_FIELDS,NUM_FIELDS\n\ndef base(**updates):\n    v={k:False for k in BOOL_FIELDS};v.update({k:0 for k in NUM_FIELDS})\n    v.update(total_clients=1000000,total_counterparts=100,daily_transactions=100000,daily_value=1000000,critical_function=True)\n    v.update(updates);return v\n\n@pytest.mark.parametrize('field,edge,above',[\n ('clients',100000,100001),('economic_loss',100000,100000.01),\n ('duration_hours',24,24.01),('downtime_minutes',120,120.01),\n ('counterparts',30,31),('transactions',10000,10001),('transaction_value',100000,100000.01),\n ('member_states',1,2)])\ndef test_thresholds(field,edge,above):\n    # Add reputation as exactly one independent category.\n    assert assess(base(media=True,**{field:edge}))['major'] is False\n    assert assess(base(media=True,**{field:above}))['major'] is True\n\ndef test_client_relative_boundary():\n    assert assess(base(total_clients=100,clients=10,media=True))['major'] is False\n    assert assess(base(total_clients=100,clients=11,media=True))['major'] is True\n\ndef test_categories_are_not_subthreshold_counts():\n    r=assess(base(clients=100001,transactions=20000,counterparts=40))\n    assert r['met_count']==1 and r['major'] is False\n    r=assess(base(duration_hours=25,downtime_minutes=121))\n    assert r['met_count']==1 and r['major'] is False\n\ndef test_gate_and_access():\n    assert assess(base(critical_function=False,media=True,economic_loss=200000))['major'] is False\n    assert assess(base(critical_function=False,regulated_service=True,media=True,economic_loss=200000))['major'] is True\n    assert assess(base(critical_function=False,malicious_access=True,access_may_cause_loss=True))['major'] is True\n    # Adverse data effect falls under 9(5)(a), so no special (b) shortcut.\n    assert assess(base(malicious_access=True,access_may_cause_loss=True,data_adverse=True))['major'] is False\n    assert assess(base(malicious_access=True,access_may_cause_loss=True,data_adverse=True,media=True))['major'] is True\n    assert assess(base(malicious_access=True,access_may_cause_loss=False))['major'] is False\n\ndef test_critical_downtime_requirement():\n    r=assess(base(critical_function=False,regulated_service=True,downtime_minutes=121,media=True))\n    assert r['major'] is False\n\ndef test_unknowns_and_zero_denominators():\n    assert assess({})['major'] is None\n    assert assess(base(media=True,clients=None))['major'] is None\n    assert assess(base(media=True,total_clients=0))['major'] is None\n    assert assess(base(media=True,total_clients=None,economic_loss=100001))['major'] is True\n    assert assess(base(critical_function=False,regulated_service=False,malicious_access=False,clients=None))['major'] is False\n\n@pytest.mark.parametrize('field,value',[('clients',-1),('economic_loss',float('nan')),('economic_loss',float('inf')),('member_states',28),('member_states',1.5),('critical_function','No'),('clients',True)])\ndef test_invalid(field,value):\n    with pytest.raises(ValueError):assess(base(**{field:value}))\n\ndef test_inconsistent_denominator():\n    with pytest.raises(ValueError):assess(base(clients=101,total_clients=100))\n\ndef test_five_scenarios():\n    data=json.loads((Path(__file__).resolve().parents[1]/'data/dora_scenarios.json').read_text())\n    assert [assess(s)['status'] for s in data]==['Major indicated','Needs review','Major indicated','Needs review','Major indicated']\n\ndef test_reporting_clocks():\n    a='2026-11-19T00:00:00Z'\n    assert deadlines(a,'2026-11-19T01:00:00Z')['initial_due_utc'].isoformat()=='2026-11-19T05:00:00+00:00'\n    assert deadlines(a,'2026-11-19T23:00:00Z')['initial_due_utc'].isoformat()=='2026-11-20T00:00:00+00:00'\n    assert deadlines(a,'2026-11-20T00:00:00Z')['initial_due_utc'].isoformat()=='2026-11-20T00:00:00+00:00'\n    assert deadlines(a,'2026-11-20T00:00:01Z')['initial_due_utc'].isoformat()=='2026-11-20T04:00:01+00:00'\n    r=deadlines(a,'2026-11-19T01:00:00Z','2026-11-19T02:00:00Z','2026-11-22T02:00:00Z')\n    assert r['intermediate_due_utc'].isoformat()=='2026-11-22T02:00:00+00:00'\n    assert r['final_due_utc'].isoformat()=='2026-12-22T02:00:00+00:00'\n\ndef test_month_end_and_timezone():\n    assert deadlines(intermediate='2028-01-31T12:00:00+01:00')['final_due_utc'].isoformat()=='2028-02-29T11:00:00+00:00'\n    assert deadlines(intermediate='2027-01-31T12:00:00Z')['final_due_utc'].day==28\n    assert deadlines()['initial_due_utc'] is None\n    with pytest.raises(ValueError):deadlines(awareness='2026-11-19T00:00:00')\n    with pytest.raises(ValueError):deadlines('2026-11-19T12:00:00Z','2026-11-19T11:00:00Z')\n", encoding="utf-8")
print('Wrote deployment metadata and regression tests')


### Test and generate deployment bundle
This cell installs the pinned packages, runs the regression tests and creates the ZIP only if all tests and required-file checks pass. No missing files are silently skipped.


In [ ]:
import subprocess, sys, zipfile
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-dev.txt'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q','--junitxml=test_results.xml'],check=True)
required = ['app.py','rules.py','prepare_data.py','requirements.txt','requirements-dev.txt','README.md','CHANGELOG_AND_VALIDATION.md','pytest.ini','test_results.xml','data/incidents.csv','data/dora_scenarios.json','data/dora_scenarios.csv','data/data_quality.json']
required += [str(p) for p in sorted(Path('tests').glob('*.py'))]
for filename in required:
    if not Path(filename).is_file():
        raise FileNotFoundError(filename)
with zipfile.ZipFile('DORA_Incident_Monitor_MVP.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for filename in required:
        z.write(filename)
print('Tests passed; deployment bundle regenerated: DORA_Incident_Monitor_MVP.zip')


In [ ]:
# Optional: download after the preceding verification cell succeeds.
# from google.colab import files
# files.download('DORA_Incident_Monitor_MVP.zip')
print('Deploy repository main branch, app.py entry point, Python 3.12; see README.md.')
